## Experiment Controls

In [ ]:
STATE_NAME = 'colorado'
ACCEPT_STRATEGY_NAME = 'neutral' # 'neutral', 'optimized', 'unoptimized'
WEIGHTS_FILE = 'test_weights'
MARKOV_STEPS = 2000
DESIRED_TCP = 0.8
CSV_FILENAME = ''

REGION_SURCHARGE = {}
GALLERY_DIR = ''
DIST_LEVEL = 'cog'


## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
# go up a level to main proj folder
parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from common import scoring
from common import plotting
from common import acceptance

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
from IPython.display import display
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
from scipy.stats import pearsonr
import networkx as nx
from gerrychain import (Partition, Graph, MarkovChain, accept)
from gerrychain.accept import always_accept
from gerrychain.proposals import recom
from gerrychain.updaters import cut_edges, county_splits
from gerrychain.tree import recursive_tree_part
from functools import partial
import json


## Process Initial Data

### Census-Level Precinct Data + Community Data

In [ ]:
vtds = gpd.read_file("data/census_vtds/co.shp")
cois = gpd.read_file("data/Colorado_communities_labeled.geojson")

# align coords
if cois.crs != vtds.crs:
    vtds = vtds.to_crs(cois.crs)

# get total presidential election 2020 vote pop
vtds['TOTVOTES20'] = vtds['PRE20D'] + vtds['PRE20R'] + vtds['PRE20O']

# calculate coi fractions per vtd
vtds['vtd_area'] = vtds.geometry.area

overlaps = gpd.overlay(vtds, cois, how='intersection')
overlaps['coi_fraction'] = overlaps.geometry.area / overlaps['vtd_area']

# drop tiny overlaps
clean_overlaps = overlaps[overlaps['coi_fraction'] > 0.01].copy()
clean_overlaps['coi_pop'] = clean_overlaps['TOTPOP'] * clean_overlaps['coi_fraction']

# build coi pop dict per vtd - Note: CO uses 'entry_ID' instead of 'cluster'
coi_dict = {}
for index, row in clean_overlaps.iterrows():
    vtd_name = row['NAME20']
    cluster_id = row['entry_ID']
    population_chunk = row['coi_pop']
    
    if vtd_name not in coi_dict:
        coi_dict[vtd_name] = {}
        
    coi_dict[vtd_name][cluster_id] = {
        'pop': population_chunk,
        'category': row.get('predicted_category', 'coi')
    }

vtds['COI_POPS'] = [coi_dict.get(name, {}) for name in vtds['NAME20']]

### Add District Data from Congressional Map - 2021

In [ ]:
if DIST_LEVEL == "cog":
    dist_2021 = gpd.read_file("data/2021_Approved_Congressional_Plan_with_Final_Adjustments/2021_Approved_Congressional_Plan_with_Final_Adjustments/2021_Approved_Congressional_Plan_w_Final_Adjustments.shp")
elif DIST_LEVEL == "ss":
    dist_2021 = gpd.read_file("data/2021_Approved_Senate_Plan_w_Final_Adjustments/2021_Approved_Senate_Plan_w_Final_Adjustments/2021_Approved_Senate_Plan_w_Final_Adjustments.shp")
elif DIST_LEVEL == "sh":
    dist_2021 = gpd.read_file("data/2021_Approved_House_Plan_w_Final_Adjustments/2021_Approved_House_Plan_w_Final_Adjustments/2021_Approved_House_Plan_w_Final_Adjustments.shp")

dist_2021 = dist_2021.to_crs(vtds.crs)

# map vtd points to congressional districts
vtd_points = vtds.copy()
vtd_points.geometry = vtd_points.representative_point()
joined_vtds_2021 = gpd.sjoin(vtd_points, dist_2021, how="left", predicate="intersects")
vtds['district_2021'] = joined_vtds_2021['District']
print(vtds.columns)

## Markov Chaining

### Init Graph and Chain

In [ ]:
vtds.geometry = vtds.geometry.buffer(0)
g = Graph.from_geodataframe(vtds)
with open(f'../weights/{WEIGHTS_FILE}.json', 'r') as f:
    state_weights = json.load(f)
g.graph['DESIRED_TCP'] = DESIRED_TCP
g.graph['WEIGHT_MAP'] = state_weights
# getting number of parts in plan
if DIST_LEVEL == "cog":
    dist_parts = 8
elif DIST_LEVEL == "ss":
    dist_parts = 35
elif DIST_LEVEL == "sh":
    dist_parts = 65
total_population = sum(node.get('TOTPOP', 1) for node in g.nodes.values())
target_pop = total_population / dist_parts
starting_assignment = recursive_tree_part(
    g, 
    parts=range(dist_parts), 
    pop_target=target_pop, 
    pop_col="TOTPOP", 
    epsilon=0.05
)
updaters_dict = { 
    '_coi_state': scoring.coi_district_pops,
    '_partisan': scoring.partisan_data, 
    'unweighted_tcp_score': scoring.calculate_unweighted_tcp,
    'weighted_tcp_score': scoring.calculate_weighted_tcp,
    'communities_split': scoring.communities_split,
    'effective_splits': scoring.effective_splits,
    'sr_entropy': scoring.square_root_entropy,
    'shannon_entropy': scoring.shannon_entropy,
    'even_splits': scoring.even_splits,
    'cut_edges': cut_edges,
    'dem_wins': scoring.count_dem_wins,
    'dem_share': scoring.dem_share,
    'dem_box': scoring.dem_boxes,
    'county_splits': county_splits('county_splits', 'COUNTYFP20'),
    'county_split_count': scoring.count_county_splits,
    'county_fragments': scoring.count_county_fragments,
    'community_fragments': scoring.community_fragments
}
initial_partition = Partition(
    g, 
    assignment=starting_assignment, 
    updaters=updaters_dict
)
proposal = partial(recom, pop_col="TOTPOP", pop_target=target_pop, epsilon=0.05, node_repeats=2, region_surcharge=REGION_SURCHARGE)
accept_function = acceptance.STRATEGIES.get(ACCEPT_STRATEGY_NAME, always_accept)
chain = MarkovChain(
    proposal=proposal,
    constraints=[],
    accept=accept_function,
    initial_state=initial_partition,
    total_steps=MARKOV_STEPS
)


### Run Chain

In [ ]:
# collect scores
num_districts = len(initial_partition.parts)

chain_results = {
    'step': [],
    'weighted_tcp_score': [],
    'unweighted_tcp_score': [],
    'communities_split': [],
    'effective_splits': [],
    'sr_entropy': [],
    'shannon_entropy': [],
    'even_splits': [],
    'cut_edges': [],
    'dem_wins': [],
    'accepted': [],
    'county_split_count': [],
    'county_fragments': [],
    'community_fragments': []
}
# district share columns
for i in range(1, num_districts + 1):
    chain_results[f'dist_{i}_dem_share'] = []

# map tracking
best_score = 0
best_map = None
worst_score = 1
worst_map = None
overall_best_score = 0
overall_best_map = None
most_seats = 0

# gallery tracking
best_overall_tcp = {'tcp': -1, 'map': None}
worst_overall_tcp = {'tcp': 999, 'map': None}

# 6 map gallery tracking
m1_val, m1_map = -9999, None  # biggest (tcp - effs)
m2_val, m2_map = 9999, None   # smallest (cs - tcp)
m3_val, m3_map = 9999, None   # smallest (tcp - effs)
m4_val, m4_map = 9999, None   # smallest (cs - effs)
m5_val, m5_map = -9999, None  # biggest (cs - effs)
m6_val, m6_map = -9999, None  # biggest (cs - tcp)

# all tcp scores grouped by seat count
tcp_scores_by_seats = {seat_count: [] for seat_count in range(num_districts + 1)}

for step, partition in enumerate(chain):
    is_accepted = 1 if (partition.parent is not None and partition is not partition.parent) else 0
    if step > 0 and step % (MARKOV_STEPS // 10 or 1) == 0:
        print(f"completed {step} steps...")

    metrics = scoring.collect_metrics(partition, [
        'weighted_tcp_score', 'unweighted_tcp_score', 'communities_split',
        'effective_splits', 'sr_entropy', 'shannon_entropy', 'even_splits',
        'dem_wins', 'dem_share', 'cut_edges', 'county_split_count', 'county_fragments', 'community_fragments'
    ])

    weighted_score = metrics['weighted_tcp_score']
    seats = metrics['dem_wins']
    shares = metrics['dem_share']
    splits = metrics['communities_split']

    chain_results['step'].append(step)
    chain_results['weighted_tcp_score'].append(weighted_score)
    chain_results['unweighted_tcp_score'].append(metrics['unweighted_tcp_score'])
    chain_results['communities_split'].append(splits)
    chain_results['effective_splits'].append(metrics['effective_splits'])
    chain_results['sr_entropy'].append(metrics['sr_entropy'])
    chain_results['shannon_entropy'].append(metrics['shannon_entropy'])
    chain_results['even_splits'].append(metrics['even_splits'])
    chain_results['cut_edges'].append(len(metrics['cut_edges']))
    chain_results['dem_wins'].append(seats)
    chain_results['accepted'].append(is_accepted)
    chain_results['county_split_count'].append(metrics['county_split_count'])
    chain_results['county_fragments'].append(metrics['county_fragments'])
    chain_results['community_fragments'].append(metrics['community_fragments'])
    # dem share per district
    for i in range(num_districts):
        chain_results[f'dist_{i+1}_dem_share'].append(shares[i])
    # append scores under its seat count 
    tcp_scores_by_seats[seats].append(weighted_score)

    # track most preserved
    if weighted_score > best_score:
        best_score = weighted_score
        best_map = partition

    # track least preserved
    if weighted_score < worst_score:
        worst_score = weighted_score
        worst_map = partition

    # track max minority seats, best score
    if seats > most_seats:
        most_seats = seats
        overall_best_score = weighted_score
        overall_best_map = partition
    elif seats == most_seats:
        if weighted_score > overall_best_score:
            overall_best_score = weighted_score
            overall_best_map = partition
            
    # track absolute best/worst overall
    if weighted_score > best_overall_tcp['tcp']:
        best_overall_tcp = {'tcp': weighted_score, 'map': partition.assignment.to_dict()}
    if weighted_score < worst_overall_tcp['tcp']:
        worst_overall_tcp = {'tcp': weighted_score, 'map': partition.assignment.to_dict()}

    # Gallery math
    t_effs = metrics['effective_splits']
    t_cs = metrics['county_split_count']
    
    if (weighted_score - t_effs) > m1_val:
        m1_val = (weighted_score - t_effs)
        m1_map = partition.assignment.to_dict()
        
    if (t_cs - weighted_score) < m2_val:
        m2_val = (t_cs - weighted_score)
        m2_map = partition.assignment.to_dict()
        
    if (weighted_score - t_effs) < m3_val:
        m3_val = (weighted_score - t_effs)
        m3_map = partition.assignment.to_dict()
        
    if (t_cs - t_effs) < m4_val:
        m4_val = (t_cs - t_effs)
        m4_map = partition.assignment.to_dict()
        
    if (t_cs - t_effs) > m5_val:
        m5_val = (t_cs - t_effs)
        m5_map = partition.assignment.to_dict()
        
    if (t_cs - weighted_score) > m6_val:
        m6_val = (t_cs - weighted_score)
        m6_map = partition.assignment.to_dict()

# save to csv
results_df = pd.DataFrame(chain_results)

if CSV_FILENAME:
    csv_filename = CSV_FILENAME
else:
    k_str = f"{int(MARKOV_STEPS/1000)}k" if MARKOV_STEPS >= 1000 else str(MARKOV_STEPS)
    base_filename = f"data/{STATE_NAME}_{ACCEPT_STRATEGY_NAME}_{WEIGHTS_FILE}_{k_str}"
    csv_filename = f"{base_filename}.csv"
    if os.path.exists(csv_filename):
        version = 1
        new_filename = f"{base_filename}_v{version}.csv"
        while os.path.exists(new_filename):
            version += 1
            new_filename = f"{base_filename}_v{version}.csv"
        csv_filename = new_filename
results_df.to_csv(csv_filename, index=False)
print(f"saved chain data to: {csv_filename}")

# save gallery maps
if GALLERY_DIR:
    import json as sys_json
    if best_overall_tcp['map']:
        with open(f"{GALLERY_DIR}best_tcp.json", 'w') as outf:
            sys_json.dump(best_overall_tcp['map'], outf)
    if worst_overall_tcp['map']:
        with open(f"{GALLERY_DIR}worst_tcp.json", 'w') as outf:
            sys_json.dump(worst_overall_tcp['map'], outf)
    if m1_map:
        with open(f"{GALLERY_DIR}1_high_tcp_low_effs.json", "w") as outf:
            sys_json.dump(m1_map, outf)
    if m2_map:
        with open(f"{GALLERY_DIR}2_high_tcp_low_cs.json", "w") as outf:
            sys_json.dump(m2_map, outf)
    if m3_map:
        with open(f"{GALLERY_DIR}3_low_tcp_high_effs.json", "w") as outf:
            sys_json.dump(m3_map, outf)
    if m4_map:
        with open(f"{GALLERY_DIR}4_low_cs_high_effs.json", "w") as outf:
            sys_json.dump(m4_map, outf)
    if m5_map:
        with open(f"{GALLERY_DIR}5_high_cs_low_effs.json", "w") as outf:
            sys_json.dump(m5_map, outf)
    if m6_map:
        with open(f"{GALLERY_DIR}6_low_tcp_high_cs.json", "w") as outf:
            sys_json.dump(m6_map, outf)
    print(f"saved gallery maps to {GALLERY_DIR}_...")


## Maps

In [ ]:
enacted_2021 = Partition(
    g,
    assignment="district_2021",
    updaters=updaters_dict
)

In [ ]:
plotting.plot_partition_detailed(enacted_2021, vtds, cois, title="2021 Enacted Map", cluster_col="entry_ID")
plotting.plot_partition_detailed(best_map, vtds, cois, title="Most Preserved Map", cluster_col="entry_ID")
plotting.plot_partition_detailed(worst_map, vtds, cois, title="Worst Preserved Map", cluster_col="entry_ID")

## Plots

In [ ]:
enacted_reference_data = {
    "2021 Enacted": {
        "score": enacted_2021["weighted_tcp_score"], 
        "wins": enacted_2021["dem_wins"],
        "county_splits": enacted_2021["county_split_count"]
    }
}
initial_score = initial_partition["weighted_tcp_score"]
initial_wins = initial_partition["dem_wins"]
initial_splits = initial_partition['county_split_count']


In [ ]:
plotting.plot_tcp_distribution(
    results_df, 
    initial_score=initial_score, 
    enacted_data=enacted_reference_data, 
    steps_str=str(MARKOV_STEPS)
)

In [ ]:
plotting.plot_dem_wins_distribution(
    results_df, 
    initial_wins=initial_wins, 
    enacted_data=enacted_reference_data, 
    steps_str=str(MARKOV_STEPS)
)

In [ ]:
plotting.plot_tcp_by_seat_count(
    results_df, 
    enacted_data=enacted_reference_data, 
    desired_tcp=DESIRED_TCP
)

In [ ]:
plotting.plot_partisan_shift(
    results_df, 
    score_col='weighted_tcp_score', 
    top_q=0.80, 
    bottom_q=0.20
)

In [ ]:

## correlation scatters
metrics_to_correlate = [
    ('communities_split', 'Comm Split'),
    ('sr_entropy', 'SRE Scores'),
    ('shannon_entropy', 'SE Scores'),
    ('even_splits', 'ES Scores')
]

sns.set_theme(style="whitegrid")
for metric_col, label in metrics_to_correlate:
    corr_coef, _ = pearsonr(results_df['weighted_tcp_score'], results_df[metric_col])
    
    plt.figure(figsize=(7, 5))
    sns.regplot(
        data=results_df, 
        x='weighted_tcp_score', 
        y=metric_col, 
        scatter_kws={"color": "blue"}, 
        line_kws={"color": "red"}, 
        marker="o"
    )
    plt.title(f"Correlation Plot (Pearson r = {corr_coef:.3f})", fontsize=14)
    plt.xlabel("Weighted TCP Scores")
    plt.ylabel(label)
    plt.show()

In [ ]:
plotting.plot_court_metrics_distributions(results_df, steps_str=str(MARKOV_STEPS))


In [ ]:
plotting.plot_county_splits_distribution(
    results_df, 
    initial_splits=initial_splits, 
    enacted_data=enacted_reference_data, 
    steps_str=str(MARKOV_STEPS)
)


In [ ]:
plotting.plot_tcp_vs_county_splits(results_df, steps_str=str(MARKOV_STEPS))
